# %% [markdown]
# # Batch ESP-r PV Output Cleaning Script
#
# This script automates the cleaning of ESP-r photovoltaic (PV) output files (`pv_res.csv`)
# across multiple buildings and model configurations.
#
# ## What it does:
# - Reads a master EPC dataset containing BUILDING_REFERENCE_NUMBER values
# - Iterates through each building folder in the baseline models directory
# - Searches for valid model subfolders (e.g., Detached_2F, SemiDetached_2F, etc.)
# - Locates `cfg/pv_res.csv` files within each model folder
# - Applies the existing cleaning process to extract and format PV power data
# - Saves a cleaned file as `pv_power.csv` in the same `cfg` directory
# - Skips buildings or model folders where `pv_res.csv` is missing
#
# ## Output:
# - Cleaned PV files saved alongside original data:
#   .../cfg/pv_power.csv
#
# ## Notes:
# - Designed for batch processing of all 117 buildings
# - Safe handling of missing or incomplete datasets
# - Logs progress for debugging and tracking

In [ ]:
# %% 
# STEP 1 — Load libraries and define file paths
# Description: Import required packages and define root dataset + model directories

import pandas as pd
import numpy as np
from pathlib import Path

# Input dataset (EPC file)
epc_path = Path("/Users/jakegehrung/Desktop/data_raw/data_processed/fintry_epc_NPV_IRR_update.csv")

# Root directory containing building folders
models_root = Path("/Users/jakegehrung/Desktop/data_raw/baseline_models")

# Read EPC dataset
epc_df = pd.read_csv(epc_path)

# Extract building IDs
building_ids = epc_df["BUILDING_REFERENCE_NUMBER"].astype(str).unique()

print(f"Total buildings found in EPC dataset: {len(building_ids)}")

Total buildings found in EPC dataset: 117


In [ ]:
# %% 
# STEP 2 — Define valid model folder names
# Description: List all possible ESP-r model folder names to check within each building directory

model_folders = [
    "SemiDetached_2F",
    "Detached_2F"
]

In [7]:
# %% 
# STEP 3 — Define PV cleaning function
# Description: Encapsulates your existing cleaning logic into a reusable function

def clean_pv_file(input_path, output_path):
    try:
        # Read raw ESP-r file
        df_raw = pd.read_csv(
            input_path,
            sep=r"\s+",
            comment="#",
            engine="python"
        )

        # Extract required columns
        df = df_raw[["Time", "atticOutput(-)"]].copy()

        # Format time
        df["Time"] = df["Time"].astype(str).str.replace("h", ":", regex=False) + ":00"
        df.columns = ["Time", "pv_power"]

        # Expected steps
        n_steps = 365 * 48

        # Create correct time index
        time_index = pd.date_range(
            start="2026-01-01 00:30:00",
            periods=n_steps,
            freq="30min"
        )
        time_str = time_index.strftime("%H:%M:%S")

        # Handle PV values
        pv_power_values = df["pv_power"].to_numpy()

        if len(pv_power_values) < n_steps:
            missing = n_steps - len(pv_power_values)
            pv_power_values = np.concatenate([pv_power_values, np.zeros(missing)])

        pv_power_values = pv_power_values[:n_steps]

        # Final dataframe
        df_clean = pd.DataFrame({
            "Time": time_str,
            "pv_power": pv_power_values
        })

        # Save output
        df_clean.to_csv(output_path, index=False)

        return True

    except Exception as e:
        print(f"Error processing {input_path}: {e}")
        return False

In [8]:
# %% 
# STEP 4 — Iterate through all buildings and process PV files
# Description: Loop over all building IDs and model folders, apply cleaning where possible

processed = 0
skipped = 0

for bldg_id in building_ids:
    building_path = models_root / bldg_id

    if not building_path.exists():
        print(f"Missing building folder: {bldg_id}")
        skipped += 1
        continue

    for model in model_folders:
        model_path = building_path / model
        cfg_path = model_path / "cfg"

        pv_input = cfg_path / "pv_res.csv"
        pv_output = models_root / bldg_id / "TOTAL" / "pv_power.csv"

        if pv_input.exists():
            success = clean_pv_file(pv_input, pv_output)

            if success:
                print(f"Processed: {pv_input}")
                processed += 1
            else:
                skipped += 1
        else:
            # Silently skip missing files (optional: uncomment print)
            # print(f"Missing pv_res.csv in: {cfg_path}")
            skipped += 1

print("\n--- SUMMARY ---")
print(f"Processed files: {processed}")
print(f"Skipped files: {skipped}")

Processed: /Users/jakegehrung/Desktop/data_raw/baseline_models/1001829067/SemiDetached_2F/cfg/pv_res.csv
Processed: /Users/jakegehrung/Desktop/data_raw/baseline_models/1001951858/Detached_2F/cfg/pv_res.csv
Processed: /Users/jakegehrung/Desktop/data_raw/baseline_models/1001829071/EndTerrace_2F/cfg/pv_res.csv
Processed: /Users/jakegehrung/Desktop/data_raw/baseline_models/1234761001/Detached_2F/cfg/pv_res.csv
Processed: /Users/jakegehrung/Desktop/data_raw/baseline_models/1001991633/Detached_1F/cfg/pv_res.csv
Processed: /Users/jakegehrung/Desktop/data_raw/baseline_models/1001664929/SemiDetached_2F/cfg/pv_res.csv
Processed: /Users/jakegehrung/Desktop/data_raw/baseline_models/1001829059/SemiDetached_2F/cfg/pv_res.csv
Processed: /Users/jakegehrung/Desktop/data_raw/baseline_models/1001063639/Detached_2F_2/cfg/pv_res.csv
Processed: /Users/jakegehrung/Desktop/data_raw/baseline_models/1234761000/Detached_2F_1/cfg/pv_res.csv
Processed: /Users/jakegehrung/Desktop/data_raw/baseline_models/1236594950